# l04 · 2일차 3교시 · XML 읽기와 KOPIS 공연 목록
**목표**: KOPIS(공연예술통합전산망)는 응답이 **XML**입니다. XML을 딕셔너리 리스트로 바꾸는 두 방법을 비교하고, 이번 주말 서울 공연 목록을 받습니다.

| 순서 | 내용 |
|---|---|
| 1 | XML이란 · 예시 XML (키 없이 연습) |
| 2 | 방법 ① ElementTree (복습, 설치 없음) |
| 3 | 방법 ② xmltodict (JSON처럼) |
| 4 | 두 방법 결과 비교 |
| 5 | 이번 주말 날짜 계산 |
| 6 | KOPIS 실제 요청 (키 필요) |

1~5번은 키 없이 실행됩니다.

> **노트북 사용법** · 셀을 클릭하고 **Shift+Enter**를 누르면 실행되고 다음 셀로 넘어갑니다.
> 셀 왼쪽의 `[1] [2]…`는 **실행한 순서**입니다. 위에서부터 차례로 실행하세요. 중간 셀을 건너뛰면 `NameError`가 납니다.
> 처음부터 깨끗하게 다시 하려면 위쪽 메뉴의 **Restart → Run All**을 누릅니다.

In [ ]:
%pip install xmltodict

## 1. XML이란
XML은 **태그**로 값을 감싸서 적는 형식입니다. JSON과 하는 일은 같고 모양만 다릅니다.

| | JSON | XML |
|---|---|---|
| 모양 | `{"prfnm": "공연 A"}` | `<prfnm>공연 A</prfnm>` |
| 여러 개 | `[ {...}, {...} ]` | 같은 태그를 반복 `<db>…</db><db>…</db>` |
| 파이썬으로 | `res.json()` 한 줄 | ElementTree 또는 xmltodict |

KOPIS 공연 목록은 `<dbs>` 안에 공연 하나마다 `<db>`가 반복되는 모양입니다.

| 태그 | 뜻 |
|---|---|
| `mt20id` | 공연 ID |
| `prfnm` | 공연명 |
| `prfpdfrom` / `prfpdto` | 공연 시작일 / 종료일 |
| `fcltynm` | 공연장 |
| `genrenm` | 장르 |

아래는 KOPIS 응답과 같은 모양의 **예시 XML**입니다. 키 없이 연습할 수 있습니다.

In [ ]:
sample = """<?xml version="1.0" encoding="UTF-8"?>
<dbs>
  <db>
    <mt20id>PF000001</mt20id>
    <prfnm>공연 A</prfnm>
    <prfpdfrom>2026.09.05</prfpdfrom>
    <prfpdto>2026.11.30</prfpdto>
    <fcltynm>공연장 가</fcltynm>
    <genrenm>뮤지컬</genrenm>
  </db>
  <db>
    <mt20id>PF000002</mt20id>
    <prfnm>공연 B</prfnm>
    <prfpdfrom>2026.10.24</prfpdfrom>
    <prfpdto>2026.10.25</prfpdto>
    <fcltynm>공연장 나</fcltynm>
    <genrenm>연극</genrenm>
  </db>
  <db>
    <mt20id>PF000003</mt20id>
    <prfnm>공연 C</prfnm>
    <prfpdfrom>2026.10.01</prfpdfrom>
    <prfpdto>2026.12.31</prfpdto>
    <fcltynm>공연장 다</fcltynm>
    <genrenm>서양음악(클래식)</genrenm>
  </db>
</dbs>"""
print(type(sample))   # 아직은 그냥 글자(str)

## 2. 방법 ① ElementTree (복습 · 설치 필요 없음)
파이썬 기본 도구입니다. 지난 수업의 따릉이 XML을 읽을 때 썼습니다.

| 코드 | 하는 일 |
|---|---|
| `ET.fromstring(글자)` | XML 글자를 읽어 맨 바깥 태그(`<dbs>`)를 돌려줌 |
| `root.findall("db")` | `<db>` 태그를 모두 찾아 리스트로 |
| `{c.tag: c.text for c in db}` | `<db>` 안의 태그 이름 → 키, 내용 → 값인 딕셔너리 |

In [ ]:
import xml.etree.ElementTree as ET   # 이름이 길어서 ET로 줄여 부름

root = ET.fromstring(sample.encode("utf-8"))   # 글자 → XML 나무 구조. root는 <dbs>

rows = []
for db in root.findall("db"):          # 공연 하나(<db>)씩
    row = {}
    for c in db:                       # <db> 안의 태그 하나씩 (<prfnm>, <fcltynm>…)
        row[c.tag] = c.text            # 태그 이름: 내용
    rows.append(row)

print(type(rows), len(rows))
print(rows[0])

> 위의 안쪽 `for` 세 줄은 슬라이드의 한 줄 `{c.tag: c.text for c in db}`(딕셔너리 컴프리헨션)과 같습니다. 짧게 쓰면 이렇습니다.

In [ ]:
rows = [{c.tag: c.text for c in db} for db in root.findall("db")]
print(rows[1])

## 3. 방법 ② xmltodict · JSON처럼 딕셔너리로
`xmltodict.parse(글자)` 한 줄이면 XML 전체가 딕셔너리가 됩니다. 1일차처럼 **대괄호로 태그 이름을 따라 내려가** 꺼냅니다.

`data["dbs"]["db"]` → `<dbs>` 안의 `<db>` 목록

**주의**: 바꾼 값은 **모두 글자(str)**입니다. 비어 있는 태그는 `None`이 됩니다.

| XML | 바꾼 값 |
|---|---|
| `<a>1</a>` | `'1'` (글자) |
| `<b></b>` | `None` |
| `<c>12,000원</c>` | `'12,000원'` |

In [ ]:
import xmltodict

data = xmltodict.parse(sample)   # XML 글자 → 딕셔너리
items = data["dbs"]["db"]        # <dbs> 안의 <db> 목록
print(type(items), len(items))
print(items[0]["prfnm"], items[0]["fcltynm"])

print(xmltodict.parse("<db><a>1</a><b></b></db>"))   # 값은 글자, 빈 태그는 None

### 주의 · 한 건만 오면 리스트가 아니라 딕셔너리
xmltodict는 `<db>`가 **하나뿐이면 리스트로 감싸지 않고 딕셔너리**로 돌려줍니다. 그러면 `for`로 돌릴 때 공연이 아니라 태그 이름이 하나씩 나와 버립니다.
그래서 `isinstance` 두 줄로 "딕셔너리면 리스트로 감싸기"를 붙입니다. (ElementTree는 `findall`이 항상 리스트라 필요 없음)

In [ ]:
one = xmltodict.parse("<dbs><db><prfnm>공연 A</prfnm></db></dbs>")
items = one["dbs"]["db"]
print(type(items))              # <class 'dict'>  ← 리스트가 아님!

if isinstance(items, dict):     # items가 딕셔너리라면
    items = [items]             # 리스트로 감싸기
print(type(items), items)

## 4. 어느 쪽이든 끝은 같은 DataFrame
| ① ElementTree | ② xmltodict |
|---|---|
| 설치 없이 쓰고 싶을 때 | JSON처럼 대괄호로 꺼내고 싶을 때 |
| 같은 태그가 반복되는 단순한 목록 | 태그가 여러 겹이어도 이름을 따라 내려감 |
| 한 건이어도 리스트 → `isinstance` 필요 없음 | 한 건일 때 `isinstance` 두 줄 필요 |

`df.equals()`로 두 표가 완전히 같은지(열 이름, 값, 순서) 확인합니다.

In [ ]:
import pandas as pd

df_et = pd.DataFrame(rows)                 # ElementTree로 만든 rows
df_xd = pd.DataFrame(data["dbs"]["db"])    # xmltodict로 만든 data
print(df_et.equals(df_xd))                 # True
df_xd

## 5. 이번 주말 날짜 계산
영화(KOBIS)는 **지난** 주말이었지만, 공연(KOPIS)은 **이번** 주말에 볼 공연을 찾습니다.

토요일의 요일 번호는 5입니다. `(5 - 오늘 요일) % 7`일을 더하면 이번 토요일입니다. (`%`는 나눈 나머지. 일요일(6)이면 `-1 % 7 = 6`일 뒤 토요일)

| 오늘 | weekday() | 더할 날 | 토요일 |
|---|---|---|---|
| 월 | 0 | 5 | 10-24 |
| 수 | 2 | 3 | 10-24 |
| 토 | 5 | 0 | 10-24 (오늘) |
| 일 | 6 | 6 | 10-31 |

날짜는 두 가지 모양으로 씁니다. KOPIS는 `"20261024"`, 날씨(Open-Meteo)는 `"2026-10-24"`.

In [ ]:
from datetime import date, timedelta

today = date.today()            # 슬라이드는 date(2026, 10, 21)
sat = today + timedelta(days=(5 - today.weekday()) % 7)   # 이번 토요일
sun = sat + timedelta(days=1)                              # 그다음 날 일요일

print(sat, sun)                                    # 날씨 API용 모양
print(sat.strftime("%Y%m%d"), sun.strftime("%Y%m%d"))   # KOPIS용 모양

## 6. KOPIS 요청 · 4단계 그대로 (키 필요)
**KOPIS 공연 목록 문서** · 주소 `http://www.kopis.or.kr/openApi/restful/pblprfr`

| 파라미터 | 뜻 | 예 |
|---|---|---|
| `service` | 발급받은 키 (KOBIS의 `key` 자리) | KOPIS_KEY |
| `stdate` | 공연 시작일 `YYYYMMDD` | 20261024 |
| `eddate` | 공연 종료일 `YYYYMMDD` (최대 31일) | 20261025 |
| `cpage` | 페이지 번호 | 1 |
| `rows` | 한 페이지 개수 (최대 100) | 10 |
| `signgucode` | 지역 코드 · 선택 | 11 (서울) |
| `shcate` | 장르 코드 · 선택 | AAAA 연극, GGGA 뮤지컬, CCCA 클래식 |
| `prfstate` | 공연 상태 · 선택 | 02 (공연중) |

**③ 확인하기**에서 KOBIS의 `faultInfo`처럼, KOPIS는 키가 틀리면 공연 대신 `returncode`·`errmsg`가 든 `<db>` 하나가 옵니다.
**④ 꺼내기**에서 `res.text` 대신 `res.content`를 넣으면 한글이 깨지지 않습니다 (XML 맨 앞에 적힌 UTF-8로 읽음).

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

# ① 주소 만들기
url = "http://www.kopis.or.kr/openApi/restful/pblprfr"
params = {"service": os.getenv("KOPIS_KEY"),          # KOPIS는 키 이름이 service
          "stdate": sat.strftime("%Y%m%d"),           # 이번 토요일부터
          "eddate": sun.strftime("%Y%m%d"),           # 이번 일요일까지
          "cpage": 1, "rows": 10,                     # 첫 페이지, 10개
          "signgucode": "11"}                         # 서울

# ② 보내기
res = requests.get(url, params=params, timeout=5)

# ③ 확인하기
res.raise_for_status()   # 키가 아예 없으면 400 오류로 여기서 멈춤

# ④ 꺼내기: XML 글자 → 딕셔너리
data = xmltodict.parse(res.content)
items = data["dbs"]["db"]
if isinstance(items, dict):        # 한 건이면 리스트로
    items = [items]

if "returncode" in items[0]:       # 키 오류 안내가 왔는지 확인
    print("KOPIS 오류:", items[0]["errmsg"])
else:
    for p in items[:3]:            # 앞 3개만 출력
        period = p["prfpdfrom"] + "~" + p["prfpdto"]
        print(p["prfnm"], period, p["fcltynm"])

## 다음 · 실습
| 파일 | 내용 |
|---|---|
| **l05_실습_공연목록_xmltodict.py** | 위 요청을 `get_performances(시작일, 종료일, 장르)` 함수로 묶은 예시 정답 |
| **l06_실습_공연목록_ElementTree.py** | 같은 함수를 ElementTree로 (XML 읽는 4줄만 다름) |
| **l07_실습_주말추천표.py** | 날씨·영화·공연을 엑셀 한 파일에 시트 세 개로 → 최종 결과물 |